In [1]:
import os
import librosa
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# Verify GPU is working
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

2026-04-02 06:22:54.405686: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775110974.620514      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775110974.684776      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775110975.137622      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775110975.137663      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775110975.137667      55 computation_placer.cc:177] computation placer alr

Num GPUs Available:  2


In [3]:
# --- CHANGE THIS PATH TO MATCH YOUR KAGGLE INPUT ---
# Example: If you named your dataset "nepali-speech", it will be "/kaggle/input/nepali-speech/raw"
DATA_PATH = "/kaggle/input/datasets/ayamkattel62/audio-dataset/raw" 
WORKING_DIR = "/kaggle/working"

SENTENCES = [
    "Ma_Khusi_Xu", "Malai_Mero_Desh_Pyaro_Lagxa", "Namaste", 
    "Ram_Le_Vaat_Khanxa", "Tapaiko_Ghar_Kaha_Xa", "TimiLai_Kasto_Chha", 
    "Uh_Mero_Mitra_Ho", "Yo_Hamro_AI_Ko_Project_ho"
]

NUM_CLASSES = len(SENTENCES)
SAMPLE_RATE = 22050
DURATION = 4.224
FRAME_LENGTH = 2048
HOP_LENGTH = 512

In [4]:
def process_audio_sequence(audio, sample_rate):
    """
    Extracts features while PRESERVING the time dimension.
    Returns shape: (time_steps, features) -> crucial for LSTMs.
    """
    # 1. Trim background noise and normalize
    audio, _ = librosa.effects.trim(audio, top_db=20)
    if len(audio) > 0:
        audio = librosa.util.normalize(audio)
        
    # 2. Pad / Truncate
    target_length = int(sample_rate * DURATION)
    if len(audio) < target_length:
        audio = np.pad(audio, (0, max(0, target_length - len(audio))), "constant")
    else:
        audio = audio[:target_length]
    
    # 3. Extract features and Transpose (.T) so time is the first axis
    zcr_val = librosa.feature.zero_crossing_rate(audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH).T
    rmse_val = librosa.feature.rms(y=audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH).T
    mfcc_val = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=13, hop_length=HOP_LENGTH).T
    
    # 4. Stack features side-by-side. 
    # Resulting shape per file: (Time_Steps, 1 + 1 + 13) = (Time_Steps, 15)
    sequence = np.hstack((zcr_val, rmse_val, mfcc_val))
    return sequence

In [5]:
X, Y = [], []

print("Extracting sequential features for LSTM...")
for category in SENTENCES:        
    folder_path = os.path.join(DATA_PATH, category)
    if not os.path.isdir(folder_path):
        print(f"Skipping {category}: directory not found")
        continue
        
    print(f"Processing category: {category}")
    count = 0
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_path = os.path.join(folder_path, file)
            try:
                audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
                features = process_audio_sequence(audio, sr)
                X.append(features)
                Y.append(category)
                count += 1
            except Exception as e:
                print(f"Error processing {file}: {e}")
                
    print(f" -> {count} samples generated.")

# Convert to Numpy Arrays
X = np.array(X)
Y = np.array(Y)
print(f"\nFinal feature shape (Samples, Time_Steps, Features): {X.shape}")

Extracting sequential features for LSTM...
Processing category: Ma_Khusi_Xu
 -> 154 samples generated.
Processing category: Malai_Mero_Desh_Pyaro_Lagxa
 -> 154 samples generated.
Processing category: Namaste
 -> 154 samples generated.
Processing category: Ram_Le_Vaat_Khanxa
 -> 154 samples generated.
Processing category: Tapaiko_Ghar_Kaha_Xa
 -> 154 samples generated.
Processing category: TimiLai_Kasto_Chha
 -> 143 samples generated.
Processing category: Uh_Mero_Mitra_Ho
 -> 154 samples generated.
Processing category: Yo_Hamro_AI_Ko_Project_ho
 -> 154 samples generated.

Final feature shape (Samples, Time_Steps, Features): (1221, 182, 15)


In [6]:
# 1. Encode Labels
le = LabelEncoder()
Y_encoded = le.fit_transform(Y)
joblib.dump(le, os.path.join(WORKING_DIR, 'label_encoder.joblib'))
Y_categorical = to_categorical(Y_encoded, num_classes=NUM_CLASSES)

# 2. Scale 3D Data for LSTM
# StandardScaler only works on 2D data, so we temporarily reshape
samples, time_steps, features = X.shape
X_reshaped = X.reshape(-1, features)

scaler = StandardScaler()
X_scaled_reshaped = scaler.fit_transform(X_reshaped)
joblib.dump(scaler, os.path.join(WORKING_DIR, 'scaler.joblib'))

# Reshape back to 3D: (Samples, Time_Steps, Features)
X_scaled = X_scaled_reshaped.reshape(samples, time_steps, features)

# 3. Train/Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y_categorical, test_size=0.2, random_state=42, stratify=Y_categorical)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (976, 182, 15)
Testing data shape: (245, 182, 15)


In [8]:
from tensorflow.keras.layers import GlobalAveragePooling1D

# 1. Build an optimized Model
model = Sequential([
    Input(shape=(time_steps, features)),
    
    # Layer 1: Reduced to 64 units to prevent overfitting on small data
    Bidirectional(LSTM(64, return_sequences=True)),
    BatchNormalization(),
    Dropout(0.2), # Lowered dropout so the network can learn faster
    
    # Layer 2: KEY FIX -> return_sequences=True
    LSTM(64, return_sequences=True), 
    
    # Layer 3: KEY FIX -> Averages the signal across all 182 time steps!
    # This prevents the network from forgetting words at the beginning of the sequence.
    GlobalAveragePooling1D(),
    
    BatchNormalization(),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(NUM_CLASSES, activation='softmax')
])

# 2. Compile
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# 3. Callbacks (Slightly relaxed)
early_stop = EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1)
lr_reduction = ReduceLROnPlateau(monitor='val_loss', patience=6, factor=0.5, min_lr=0.00001, verbose=1)

# 4. Train
history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, lr_reduction]
)

# 5. Save Model
model_path = os.path.join(WORKING_DIR, 'nepali_lstm_model.h5')
model.save(model_path)
print(f"Model saved to {model_path}")

Epoch 1/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.3479 - loss: 1.9068 - val_accuracy: 0.4449 - val_loss: 1.9468 - learning_rate: 0.0010
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.8165 - loss: 0.7637 - val_accuracy: 0.7224 - val_loss: 1.7860 - learning_rate: 0.0010
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9024 - loss: 0.4086 - val_accuracy: 0.8122 - val_loss: 1.5828 - learning_rate: 0.0010
Epoch 4/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9620 - loss: 0.2158 - val_accuracy: 0.8612 - val_loss: 1.3825 - learning_rate: 0.0010
Epoch 5/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9788 - loss: 0.1314 - val_accuracy: 0.9551 - val_loss: 1.0911 - learning_rate: 0.0010
Epoch 6/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9788 - loss: 0.1090 - val_accuracy: 0.9306 - val_loss: 0.8858 - learning_rate: 0.0010
Epoch 7/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9797 - loss: 0.0893 - 

Model saved to /kaggle/working/nepali_lstm_model.h5


In [9]:
import shutil

# Zip the saved files so you can download them easily to your local machine
shutil.make_archive(os.path.join(WORKING_DIR, 'nepali_model_artifacts'), 
                    'zip', 
                    WORKING_DIR, 
                    logger=None)

print("✅ Training complete! All artifacts (Model, Scaler, Encoder) have been zipped.")
print("-> Look in the 'Output' section of Kaggle on the right sidebar to download 'nepali_model_artifacts.zip'.")

✅ Training complete! All artifacts (Model, Scaler, Encoder) have been zipped.
-> Look in the 'Output' section of Kaggle on the right sidebar to download 'nepali_model_artifacts.zip'.
